# Libraries

In [1]:
# Reinstall PyTorch stack with CUDA 12.8 support
!pip uninstall -y torch torchvision torchaudio torchao
!pip install -q \
    torch==2.10.0+cu128 \
    torchvision==0.25.0+cu128 \
    torchaudio==2.10.0+cu128 \
    torchao==0.10.0 \
    --index-url https://download.pytorch.org/whl/cu128

# vLLM
!pip install -q vllm==0.18.0

# Dependencies
!pip install -q "protobuf>=5.26.1,<6.0.0" --break-system-packages

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 85.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 57.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.

In [2]:
# Standard library imports
import os
import gc
import json
import time
import random
import shutil
import subprocess

# Third-party libraries
import numpy as np
from vllm import LLM, SamplingParams
from huggingface_hub import login
from transformers import AutoTokenizer, set_seed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download

# Deep learning framework
import torch

2026-05-25 00:34:41.624828: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779669281.856593      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779669281.920912      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779669282.434812      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779669282.434860      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779669282.434863      57 computation_placer.cc:177] computation placer alr

# Global Variables

In [3]:
# User & Repository Configuration
USER_NAME = "Abdelrahmanemam01"
EMAIL     = "abdulrahmanemam01@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Llama-3.2-3B-Instruct"

# Model & Tokenizer Configuration
MODEL_ID     = "EdgeCompress01/Llama-3.2-3B-Instruct-WANDA-GPTQ-4bit"
TOKENIZER_ID = "meta-llama/Llama-3.2-3B-Instruct"

MODEL_NAME           = "Llama-3.2-3B-Instruct"
MODEL_NAME_IN_REPO   = "Llama-3.2-3B-Instruct-WANDA-GPTQ"
COMPRESSION_METHOD   = "WANDA & GPTQ"
MODEL_TYPE           = "Pruning & Quantization"
SPARSITY = 25
PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [4]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [5]:
set_reproducibility(SEED)

# Huggingface Login

In [6]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [7]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Download Model

In [8]:
local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Models/25/model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Models/25/tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

quant_log.csv: 0.00B [00:00, ?B/s]

quantize_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/183 [00:00<?, ?B/s]

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [9]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

**RealPrompt Tokenization**

In [10]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [11]:
llm = LLM(
    model=f"{local_path}/{PATH}",
    tokenizer=TOKENIZER_ID,
    dtype="float16",
    max_model_len=256,
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    attention_backend = "TRITON_ATTN",
    disable_log_stats=False,
    enable_prefix_caching = False
)
print("vLLM Initialized Successfully!")


sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

ttft_sampling_params = SamplingParams(
    temperature=0.0,
    seed = SEED,
    max_tokens=1,
    ignore_eos=True # Ensures it doesn't accidentally stop early
)

INFO 05-25 00:35:22 [utils.py:233] non-default args: {'tokenizer': 'meta-llama/Llama-3.2-3B-Instruct', 'dtype': 'float16', 'max_model_len': 256, 'tensor_parallel_size': 2, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.85, 'attention_backend': 'TRITON_ATTN', 'model': '/kaggle/working/model/Models/25'}
INFO 05-25 00:35:44 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 05-25 00:35:44 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 05-25 00:35:44 [model.py:1582] Using max model len 256
INFO 05-25 00:35:44 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 05-25 00:35:44 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-25 00:35:44 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 05-25 00:35:45 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/

2026-05-25 00:35:57.570852: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779669357.596033     753 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779669357.603404     753 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779669357.620515     753 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779669357.620562     753 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779669357.620568     753 computation_placer.cc:177] computation placer alr

(EngineCore pid=753) INFO 05-25 00:36:06 [core.py:103] Initializing a V1 LLM engine (v0.18.0) with config: model='/kaggle/working/model/Models/25', speculative_config=None, tokenizer='meta-llama/Llama-3.2-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=256, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=gptq_marlin, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoin

2026-05-25 00:36:11.311829: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-25 00:36:11.312104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779669371.336955     779 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779669371.337041     778 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779669371.344311     779 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1779669371.344443     778 cuda_blas.cc:1

(Worker pid=778) INFO 05-25 00:36:23 [parallel_state.py:1395] world_size=2 rank=0 local_rank=0 distributed_init_method=tcp://127.0.0.1:37543 backend=nccl
(Worker pid=779) INFO 05-25 00:36:23 [parallel_state.py:1395] world_size=2 rank=1 local_rank=1 distributed_init_method=tcp://127.0.0.1:37543 backend=nccl


...

(Worker pid=778) INFO 05-25 00:36:23 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=778) WARNING 05-25 00:36:24 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=779) WARNING 05-25 00:36:24 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=778) WARNING 05-25 00:36:24 [custom_all_reduce.py:165] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=779) WARNING 05-25 00:36:24 [custom_all_reduce.py:165] Custom allreduce is disabled because your platform lacks GPU P2P capability or P2P test failed. To silence this warning, specify disable_custom_all_reduce=True explicitly.
(Worker pid=779) INFO 05-25 00:36:24 [parallel_state.py:1717] rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 1, EP ra

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.03s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.03s/it]
(Worker_TP0 pid=778) 


(Worker_TP0 pid=778) INFO 05-25 00:36:27 [default_loader.py:384] Loading weights took 1.08 seconds
(Worker_TP0 pid=778) INFO 05-25 00:36:28 [gpu_model_runner.py:4566] Model loading took 1.08 GiB memory and 1.612043 seconds
(Worker_TP0 pid=778) INFO 05-25 00:36:40 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/01b2283c56/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=778) INFO 05-25 00:36:40 [backends.py:1048] Dynamo bytecode transform time: 10.92 s
(Worker_TP0 pid=778) INFO 05-25 00:36:47 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP1 pid=779) INFO 05-25 00:36:47 [backends.py:371] Cache the graph of compile range (1, 8192) for later use
(Worker_TP0 pid=778) INFO 05-25 00:36:53 [backends.py:387] Compiling a graph for compile range (1, 8192) takes 11.82 s
(Worker_TP0 pid=778) INFO 05-25 00:36:55 [decorators.py:627] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/0164

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 10.70it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:06<00:00,  5.50it/s]


(Worker_TP0 pid=778) INFO 05-25 00:37:26 [gpu_model_runner.py:5746] Graph capturing finished in 12 secs, took 1.00 GiB
(Worker_TP0 pid=778) INFO 05-25 00:37:26 [gpu_worker.py:617] CUDA graph pool memory: 1.0 GiB (actual), 1.09 GiB (estimated), difference: 0.09 GiB (9.0%).
(Worker_TP1 pid=779) INFO 05-25 00:37:26 [gpu_worker.py:617] CUDA graph pool memory: 1.0 GiB (actual), 1.09 GiB (estimated), difference: 0.09 GiB (9.0%).
(EngineCore pid=753) INFO 05-25 00:37:26 [core.py:281] init engine (profile, create kv cache, warmup model) took 57.71 seconds
(EngineCore pid=753) INFO 05-25 00:37:27 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 05-25 00:37:27 [llm.py:391] Supported tasks: ['generate']
vLLM Initialized Successfully!


# Inference

In [12]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times      = []
latency_times   = []
decode_times    = []
inference_times = []  # prefill + decode

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    ttft      = metrics.first_token_ts - metrics.scheduled_ts
    latency   = metrics.last_token_ts  - metrics.queued_ts
    decode    = metrics.last_token_ts  - metrics.first_token_ts
    inference = ttft + decode                                   # prefill + decode

    ttft_times.append(ttft)
    latency_times.append(latency)
    decode_times.append(decode)
    inference_times.append(inference)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr, latency_arr, decode_arr, inference_arr = (
    np.array(ttft_times),
    np.array(latency_times),
    np.array(decode_times),
    np.array(inference_times),
)

ttft_mean,      ttft_std      = ttft_arr.mean(),      ttft_arr.std()
latency_mean,   latency_std   = latency_arr.mean(),   latency_arr.std()
inference_mean, inference_std = inference_arr.mean(), inference_arr.std()

# decode throughput: excludes first token, over decode-only window
decode_throughput_arr                           = (MAX_GENERATION_TOKENS - 1) / decode_arr
decode_throughput_mean, decode_throughput_std   = (
    decode_throughput_arr.mean(), decode_throughput_arr.std()
)

# overall throughput: all tokens over e2e latency
overall_throughput_arr                          = MAX_GENERATION_TOKENS / latency_arr
overall_throughput_mean, overall_throughput_std = (
    overall_throughput_arr.mean(), overall_throughput_arr.std()
)

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

INFO 05-25 00:37:39 [loggers.py:259] Engine 000: Avg prompt throughput: 51.9 tokens/s, Avg generation throughput: 113.1 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-25 00:37:49 [loggers.py:259] Engine 000: Avg prompt throughput: 45.0 tokens/s, Avg generation throughput: 130.2 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
INFO 05-25 00:37:59 [loggers.py:259] Engine 000: Avg prompt throughput: 45.0 tokens/s, Avg generation throughput: 129.8 tokens/s, Running: 1 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%


In [13]:
print(latency_times)

[1.1386639469997135, 1.1455313900000874, 1.1401566910003567, 1.1394678999999996, 1.1416225090001717, 1.1422880660002193, 1.147924317000161, 1.1504359979999208, 1.1474617599997146, 1.1501682819998678, 1.1486935459997767, 1.144422115999987, 1.149127161999786, 1.151458165999884, 1.1517870489997222, 1.1514255020001656, 1.1510246980001284, 1.1534522460001426, 1.1560524550000082, 1.1490080609996767, 1.1603554440002881, 1.149658215999807, 1.150259793999794, 1.153404514000158, 1.153463057999943, 1.1510848739999346, 1.1550049729999046, 1.155568907000088, 1.156742228999974, 1.1538399500000196]


In [14]:
print(ttft_times)

[0.02256948200010811, 0.02770443899999009, 0.021826010999575374, 0.02286685099988972, 0.02270277699972212, 0.023396479999973963, 0.029407999999875756, 0.02921135399992636, 0.02759756900013599, 0.028507075000106852, 0.027735708999898634, 0.021568492999904265, 0.02728633400010949, 0.027744686999994883, 0.028383458999996947, 0.026890862000072957, 0.026891889000125957, 0.02771114300003319, 0.028760035999766842, 0.02452168999980131, 0.032935145999999804, 0.022904897999978857, 0.022539510999649792, 0.02521090699974593, 0.025635699999838835, 0.021534609000354976, 0.023706374000084907, 0.02614311999968777, 0.025150722000034875, 0.0220675279997522]


# Results

**Writing in json file**

In [15]:
benchmark_results = {
    "model_name"                        : MODEL_NAME,
    "model_type"                       : MODEL_TYPE,
    "compression_method"               : COMPRESSION_METHOD,
    "sparsity"                         : SPARSITY,
    "input_token_count"                 : input_tokens,
    "generated_token_count"             : total_generated_tokens,
    "ttft_sec"                          : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "inference_sec"                     : f"{inference_mean:.2f} ± {inference_std:.2f}",
    "latency_sec"                       : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "decode_throughput_tokens_per_sec"  : f"{decode_throughput_mean:.2f} ± {decode_throughput_std:.2f}",
    "overall_throughput_tokens_per_sec" : f"{overall_throughput_mean:.2f} ± {overall_throughput_std:.2f}",
}

# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

[INFO] Metrics successfully saved to 'model_metrics.json'


**Push Results to GitHub Repository**

In [17]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}/Sparsity/{SPARSITY}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")

Cloning repository into: /kaggle/working/temp_git_repo


Cloning into '/kaggle/working/temp_git_repo'...


[main 06fe004] Added the cloud inference evaluation results to Evaluations/InferenceEvaluations/CloudEvaluation/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-WANDA-GPTQ/Sparsity/25
 1 file changed, 13 insertions(+)
 create mode 100644 Evaluations/InferenceEvaluations/CloudEvaluation/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-WANDA-GPTQ/Sparsity/25/model_metrics.json
Successfully pushed to 'MahmoudOsama20/EdgeCompress' → 'Evaluations/InferenceEvaluations/CloudEvaluation/Llama-3.2-3B-Instruct/Llama-3.2-3B-Instruct-WANDA-GPTQ/Sparsity/25'


To https://github.com/MahmoudOsama20/EdgeCompress.git
   c5cf343..06fe004  main -> main
